# T2.1 – DBRepo Schema Setup
**Vienna Weather Wet-Month Prediction Experiment**

This notebook creates the database and tables in DBRepo via the Python REST client,
and adds descriptive metadata to make the database citable.

**Source dataset:** Stadt Wien. *Wetter seit 1872 Hohe Warte Wien*. data.gv.at, CC BY 4.0.  
**Original publisher:** Stadt Wien / MA 23 (https://www.data.gv.at)  
**License:** CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/)  
**Dataset URL:** https://www.data.gv.at/datasets/69a06550-1ede-4f50-9c36-e7fb5cf6e7e8

## 0. Install & import dependencies

In [1]:
# Install the DBRepo Python client (match version shown in bottom-left of the DBRepo UI)
!pip install dbrepo==1.13.4 pandas requests --quiet

In [2]:
import pandas as pd
import requests
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    FilterDefinition,
    FilterType,
    OrderDefinition,
    OrderType,
)
import pandas as pd
import time



## 1. Configuration
Fill in your DBRepo credentials and the container ID before running.
The container ID can be found in the DBRepo UI under *Admin → Containers*.

In [3]:
# ── EDIT THESE ──────────────────────────────────────────────
ENDPOINT  = "https://test.dbrepo.tuwien.ac.at"
USERNAME  = "azra1558"   
PASSWORD  = "Katalizator1558!"   

DATABASE_NAME    = "vienna_weather_wet_months"
CSV_URL          = "https://www.wien.gv.at/data/ogd/ma23/vie-bdl-ecl-wea-1872f.csv"
client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)


In [4]:
print("Current user:", client.whoami())


azra1558
Current user: azra1558


## 2. Connect to DBRepo

In [5]:

# Verify connection by listing existing databases
dbs = client.get_databases()
print(f"Connected. Found {len(dbs)} existing database(s).")

Connected. Found 5 existing database(s).


## 3. Create the database

In [6]:
containers = client.get_containers()
print(containers)
CONTAINER_ID = containers[0].id

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [7]:
db = client.create_database(
    container_id=CONTAINER_ID,
    name=DATABASE_NAME,
    is_public=False
)

print(db)

ValidationError: 3 validation errors for Database
is_dashboard_enabled
  Field required [type=missing, input_value={'id': '899bfcba-7fec-40c..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
container
  Field required [type=missing, input_value={'id': '899bfcba-7fec-40c..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
owner
  Field required [type=missing, input_value={'id': '899bfcba-7fec-40c..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

The validation error from above can be ignored, everything was created it was only some issue here with parsing.

In [8]:
dbs = client.get_databases()

for db in dbs:
    print(db.id, db.name)

DATABASE_ID = [db.id for db in dbs if db.name == DATABASE_NAME][0]
print("DATABASE_ID:", DATABASE_ID)   

899bfcba-7fec-40c9-9076-3a3a9372c844 vienna_weather_wet_months
412fb0ce-5299-4d0e-a271-4641b1365b8a data_stewardship_g12_unemployment_prediction
38707917-e942-45c3-a3dd-d2bfc1c106af Vienna Demographic Forecasting
c77e4bfb-7f16-4a47-924b-430778476562 vienna_weather_wet_months
81d82941-cae0-4cba-b27d-1bd883dd713a data_stew_grp22_air_quality
17de844a-62fc-406f-abb2-fffb22b44d01 meine-datenbank
DATABASE_ID: 899bfcba-7fec-40c9-9076-3a3a9372c844


## 4. Create tables in DBRepo and upload data

In [22]:
df_station = pd.DataFrame([{
    "station_num": 5901,
    "nuts_code": "AT13",
    "district_code": 91900,
    "sub_district_code": 91905,
    "station_name": "Wien - Hohe Warte",
    "latitude_deg": 48.248611,
    "longitude_deg": 16.356944,
    "altitude_m": 202.0
}]).set_index("station_num")

existing_station_table = None

for table in client.get_tables(DATABASE_ID):
    if table.name == "station":
        existing_station_table = table
        break

if existing_station_table is not None:
    table_station = existing_station_table
    print("station already exists:", table_station.id)
else:
    table_station = client.create_table(
        database_id=DATABASE_ID,
        name="station",
        dataframe=df_station,
        is_public=False,
        is_schema_public=False,
        description="Station metadata (Hohe Warte, Vienna). Source: Stadt Wien, CC BY 4.0",
        with_data=False
    )
    print("station table created:", table_station.id)

station already exists: e6779029-ce40-4a9a-ad17-147e183dc757


In [24]:
df_time = pd.DataFrame([
    {"time_id": 1, "ref_year": 2020, "ref_month": 1},
    {"time_id": 2, "ref_year": 2020, "ref_month": 2}
]).set_index("time_id")

existing_time_table = None

for table in client.get_tables(DATABASE_ID):
    if table.name == "time_dimension":
        existing_time_table = table
        break

if existing_time_table is not None:
    table_time = existing_time_table
    print("time_dimension already exists:", table_time.id)
else:
    table_time = client.create_table(
        database_id=DATABASE_ID,
        name="time_dimension",
        dataframe=df_time,
        is_public=False,
        is_schema_public=False,
        description="Time dimension (year, month)",
        with_data=False
    )
    print("time_dimension created:", table_time.id)

time_dimension already exists: 9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2


In [15]:
import numpy as np
df_weather_full = pd.DataFrame([{
    "measurement_id": 1,
    "station_num": 5901,
    "time_id": 1,
    "t_mean_c": 0.1, "t_max_c": 0.1, "t_min_c": 0.1,
    "mean_t_max_c": 0.1, "mean_t_min_c": 0.1,
    "p_mean_hpa": 0.1, "p_max_hpa": 0.1, "p_min_hpa": 0.1,
    "precp_sum_mm": 0.1,
    "num_precp_01": 5,
    "rel_hum_pct": 0.1, "rel_hum_max_pct": 0.1, "rel_hum_min_pct": 0.1,
    "wind_vel_ms": 0.1, "wind_vel_max_ms": 0.1,
    "num_wind_vel60": 5,
    "sun_h": 0.1,
    "num_clear": 5, "num_cloud": 5,
    "num_frost": 5, "num_ice": 5, "num_summer": 5, "num_heat": 5
}])

int_cols = ["measurement_id", "station_num", "time_id",
            "num_precp_01", "num_wind_vel60", "num_clear",
            "num_cloud", "num_frost", "num_ice", "num_summer", "num_heat"]
for col in int_cols:
    df_weather_full[col] = df_weather_full[col].astype(np.int64)

df_weather_full = df_weather_full.set_index("measurement_id")
print(df_weather_full.dtypes)

station_num          int64
time_id              int64
t_mean_c           float64
t_max_c            float64
t_min_c            float64
mean_t_max_c       float64
mean_t_min_c       float64
p_mean_hpa         float64
p_max_hpa          float64
p_min_hpa          float64
precp_sum_mm       float64
num_precp_01         int64
rel_hum_pct        float64
rel_hum_max_pct    float64
rel_hum_min_pct    float64
wind_vel_ms        float64
wind_vel_max_ms    float64
num_wind_vel60       int64
sun_h              float64
num_clear            int64
num_cloud            int64
num_frost            int64
num_ice              int64
num_summer           int64
num_heat             int64
dtype: object


In [18]:
# Create weather_measurement table only if it does not already exist

existing_weather_table = None

for table in client.get_tables(DATABASE_ID):
    if table.name == "weather_measurement":
        existing_weather_table = table
        break

if existing_weather_table is not None:
    table_weather = existing_weather_table
    print("weather_measurement already exists:", table_weather.id)
else:
    table_weather = client.create_table(
        database_id=DATABASE_ID,
        name="weather_measurement",
        dataframe=df_weather_full,
        is_public=False,
        is_schema_public=False,
        with_data=False
    )
    print("Created:", table_weather.id)

weather_measurement already exists: 2212bed4-ef8f-4d95-bb65-20b2adb28abd


## 5. Verify – print summary

In [19]:
db = client.get_database(DATABASE_ID)
tables = client.get_tables(DATABASE_ID)

print("=" * 50)
print(f"Database : {db.name}  (id: {db.id})")
print(f"Tables   : {[t.name for t in tables]}")

for t in tables:
    count = client.get_table_data_count(DATABASE_ID, t.id)
    print(f"  {t.name}: {count} rows")

print("=" * 50)
print("T2.1 complete. The DB and Table IDs are:")
print(f"DATABASE_ID             : {DATABASE_ID}")
for t in tables:
    print(t.name, t.id)

Database : vienna_weather_wet_months  (id: 899bfcba-7fec-40c9-9076-3a3a9372c844)
Tables   : ['weather_measurement', 'time_dimension', 'station']
  weather_measurement: 0 rows
  time_dimension: 0 rows
  station: 0 rows
T2.1 complete. The DB and Table IDs are:
DATABASE_ID             : 899bfcba-7fec-40c9-9076-3a3a9372c844
weather_measurement 2212bed4-ef8f-4d95-bb65-20b2adb28abd
time_dimension 9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2
station e6779029-ce40-4a9a-ad17-147e183dc757


In [25]:
tables = client.get_tables(DATABASE_ID)

for t in tables:
    table = client.get_table(DATABASE_ID, t.id)
    print(f"\nTable: {table.name} ({table.id})")
    for col in table.columns:
        print(f"  - {col.name}: {col.type}")


Table: weather_measurement (2212bed4-ef8f-4d95-bb65-20b2adb28abd)
  - measurement_id: ColumnType.BIGINT
  - station_num: ColumnType.BIGINT
  - time_id: ColumnType.BIGINT
  - t_mean_c: ColumnType.DECIMAL
  - t_max_c: ColumnType.DECIMAL
  - t_min_c: ColumnType.DECIMAL
  - mean_t_max_c: ColumnType.DECIMAL
  - mean_t_min_c: ColumnType.DECIMAL
  - p_mean_hpa: ColumnType.DECIMAL
  - p_max_hpa: ColumnType.DECIMAL
  - p_min_hpa: ColumnType.DECIMAL
  - precp_sum_mm: ColumnType.DECIMAL
  - num_precp_01: ColumnType.BIGINT
  - rel_hum_pct: ColumnType.DECIMAL
  - rel_hum_max_pct: ColumnType.DECIMAL
  - rel_hum_min_pct: ColumnType.DECIMAL
  - wind_vel_ms: ColumnType.DECIMAL
  - wind_vel_max_ms: ColumnType.DECIMAL
  - num_wind_vel60: ColumnType.BIGINT
  - sun_h: ColumnType.DECIMAL
  - num_clear: ColumnType.BIGINT
  - num_cloud: ColumnType.BIGINT
  - num_frost: ColumnType.BIGINT
  - num_ice: ColumnType.BIGINT
  - num_summer: ColumnType.BIGINT
  - num_heat: ColumnType.BIGINT

Table: time_dimension (9f